# Diviser pour régner — Exercice 3 : l'exponentiation rapide (correction)

Ce notebook contient, pour chaque question, trois blocs **toujours séparés** :

| Bloc | Contenu |
|:---|:---|
| 📝 **ÉNONCÉ** | la question, recopiée du sujet |
| ✅ **RÉSOLUTION** | la réponse attendue (code et/ou texte) |
| 💡 **EXPLICATION PÉDAGOGIQUE** | pourquoi ça marche, le lien avec le cours, les erreurs à éviter. **N'est pas à recopier.** |

**Notions travaillées :** compter les opérations d'un algorithme, récursivité, éviter de **recalculer** deux fois la même chose.

Exécuter les cellules **dans l'ordre** (*Cell → Run All*).

Pour vérifier les réponses, on compte les multiplications grâce à une fonction `mult` et une variable globale `nb_mult` (non demandé).

---
# QUESTION 1

#### 📝 ÉNONCÉ

> On souhaite calculer $x^n$, où n est un entier ⩾ 1 et où x peut désigner un nombre réel, une fraction, une matrice et plus généralement tout élément d'un ensemble muni d'une loi de multiplication.
> L'approche par « force brute » consiste à écrire que $x^n = x \cdots x$ (n facteurs), et conduit à la fonction suivante.
> ```python
> def power_bf(x, n):
>     p = 1
>     for i in range(n):
>         p = p*x
>     return p
> ```
> Dans cet algorithme, combien de multiplications sont effectuées pour calculer $x^n$ ?

#### ✅ RÉSOLUTION

**$n$ multiplications** : la boucle tourne $n$ fois et fait une multiplication à chaque tour.

In [1]:
nb_mult = 0

def mult(a, b):
    global nb_mult
    nb_mult += 1
    return a * b


def power_bf(x, n):
    p = 1
    for i in range(n):
        p = mult(p, x)
    return p


nb_mult = 0
print(power_bf(2, 100), "-> multiplications :", nb_mult)

1267650600228229401496703205376 -> multiplications : 100


#### 💡 EXPLICATION PÉDAGOGIQUE

- Le coût est **linéaire** : doubler $n$ double le travail.
- On pourrait en économiser une en partant de `p = x`, ce qui fait $n - 1$.

---
# QUESTION 2

#### 📝 ÉNONCÉ

> Les propriétés des puissances permettent de remarquer que
> $x^n = \begin{cases} x & \text{si } n = 1 \\ x^{n/2}\cdot x^{n/2} & \text{si } n \text{ est pair} \\ x \cdot x^{n-1} & \text{si } n \text{ est impair} \end{cases}$
> La méthode « diviser pour régner » peut donc s'appliquer dans ce contexte, et il mène à l'algorithme appelé exponentiation rapide, dont on donne ci-dessous une version récursive particulièrement maladroite.
> ```python
> def power_dac(x, n):
>     if n==1 :
>         return x
>     if n%2==0 :
>         return power_dac(x, n//2)*power_dac(x, n//2)
>     else :
>         return x*power_dac(x, n-1)
> ```
> (a) Expliquer pourquoi la ligne 5 est, quoique correcte, une mauvaise idée en termes de performances.
> (b) Corriger cet algorithme en conséquence.
> (c) Déterminer le coût, en nombre de multiplications, de l'algorithme corrigé pour $n = 100$.

## 2 (a)

#### ✅ RÉSOLUTION

La ligne 5 calcule **deux fois exactement la même chose**, $x^{n/2}$, et chacun de ces appels refait lui-même deux appels identiques, etc. Le nombre d'appels **double à chaque niveau** : on obtient environ $n$ multiplications, autant que la force brute.

In [2]:
def power_dac(x, n):
    if n == 1:
        return x
    if n % 2 == 0:
        return mult(power_dac(x, n//2), power_dac(x, n//2))
    else:
        return mult(x, power_dac(x, n-1))


nb_mult = 0
power_dac(2, 100)
print("Version maladroite, n = 100 :", nb_mult, "multiplications")

Version maladroite, n = 100 : 99 multiplications


#### 💡 EXPLICATION PÉDAGOGIQUE

- Avec $\log(n)$ niveaux et un nombre d'appels qui double à chaque niveau, on obtient environ $2^{\log(n)} = n$ appels. Diviser le problème n'a servi à rien.
- Le comptage le confirme : 99 multiplications pour $n = 100$, contre 100 pour la force brute.

## 2 (b)

#### ✅ RÉSOLUTION

On calcule $x^{n/2}$ **une seule fois**, on le **stocke dans une variable**, puis on l'élève au carré :

In [3]:
def power_dac(x, n):
    if n == 1:
        return x
    if n % 2 == 0:
        y = power_dac(x, n//2)        # un seul appel récursif
        return mult(y, y)
    else:
        return mult(x, power_dac(x, n-1))


nb_mult = 0
print(power_dac(2, 100) == 2**100, "-> multiplications :", nb_mult)

True -> multiplications : 8


#### 💡 EXPLICATION PÉDAGOGIQUE

Une seule ligne change : `y = power_dac(x, n//2)` puis `return y*y`. Il n'y a plus qu'**un** appel récursif à chaque étape.

## 2 (c)

#### ✅ RÉSOLUTION

| $n$ | parité | appel suivant | multiplication |
|:---|:---|:---|:---|
| 100 | pair | $x^{50}$ | $y \cdot y$ |
| 50 | pair | $x^{25}$ | $y \cdot y$ |
| 25 | impair | $x^{24}$ | $x \cdot (\ldots)$ |
| 24 | pair | $x^{12}$ | $y \cdot y$ |
| 12 | pair | $x^{6}$ | $y \cdot y$ |
| 6 | pair | $x^{3}$ | $y \cdot y$ |
| 3 | impair | $x^{2}$ | $x \cdot (\ldots)$ |
| 2 | pair | $x^{1}$ | $y \cdot y$ |
| 1 | — | cas de base | aucune |

**Total : 8 multiplications** pour $n = 100$ (contre 100 pour la force brute et 99 pour la version maladroite).

*Comparaison (non demandée) pour plusieurs valeurs de n :*

In [4]:
print(f"{'n':>8} | {'force brute':>11} | {'maladroite':>10} | {'rapide':>6}")


def power_maladroite(x, n):
    if n == 1:
        return x
    if n % 2 == 0:
        return mult(power_maladroite(x, n//2), power_maladroite(x, n//2))
    return mult(x, power_maladroite(x, n-1))


for n in [10, 100, 1000, 10000]:
    res = []
    for f in [power_bf, power_maladroite, power_dac]:
        nb_mult = 0
        f(1, n)
        res.append(nb_mult)
    print(f"{n:>8} | {res[0]:>11} | {res[1]:>10} | {res[2]:>6}")

       n | force brute | maladroite | rapide
      10 |          10 |          9 |      4
     100 |         100 |         99 |      8
    1000 |        1000 |        999 |     14
   10000 |       10000 |       9999 |     17


#### 💡 EXPLICATION PÉDAGOGIQUE

- **Méthode :** on suit les appels à la main ; chaque ligne du tableau correspond à **une** multiplication, sauf le cas de base.
- **En général :** chaque étape « pair » divise $n$ par 2, et une étape « impair » est toujours suivie d'une étape « pair ». Il y a donc au plus $2\log(n)$ multiplications : le coût est **logarithmique**.
- **À retenir :** dans un algorithme « diviser pour régner », il faut que les sous-problèmes soient **différents**. Si on résout deux fois le même sous-problème, on perd tout le bénéfice de la division.